# Image to text từ structured corpus trên Kaggle

Notebook đọc `structured_corpus.jsonl` và thư mục ảnh, mô tả từng block `figure` bằng VLM, rồi tổng hợp kết quả vào một file JSON độc lập. Kết quả được lưu sau từng ảnh nên có thể chạy lại để tiếp tục; notebook không sửa hoặc sao chép structured corpus.

Trước khi chạy: bật GPU trong **Settings → Accelerator**, bật Internet nếu tải model từ Hugging Face, rồi điền `STRUCTURED_CORPUS_PATH` và `IMAGES_DIR` ở ô cấu hình.

In [ ]:
%%capture
%pip install -q -U "transformers>=5.2,<6" "accelerate>=1.1,<2" "bitsandbytes>=0.49,<1" "Pillow>=10,<13" "safetensors>=0.4,<1"

## 1. Cấu hình

Điền đường dẫn đến `structured_corpus.jsonl` và thư mục ảnh sau khi thêm dataset vào Kaggle. Có thể đặt `LIMIT = 3` để thử nhanh, sau đó đổi thành `None` để chạy toàn bộ. Đặt `RETRY_ERRORS = True` khi muốn thử lại các block đã có record lỗi. Khi tiếp tục ở phiên Kaggle mới, gắn file output cũ thành dataset rồi điền đường dẫn read-only đó vào `RESUME_PATH`.

`MAX_ARTICLE_CHARS` giới hạn ngữ cảnh structured quanh đúng placeholder của ảnh mục tiêu để giảm thời gian và VRAM; đặt `None` nếu muốn dùng toàn bộ article. Nếu model được gắn dưới dạng Kaggle Dataset, đổi `MODEL_ID` thành đường dẫn local của model.

In [ ]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path

STRUCTURED_CORPUS_PATH = ""
IMAGES_DIR = ""
RESUME_PATH = ""  # Tùy chọn: checkpoint JSON từ /kaggle/input của phiên trước
OUTPUT_PATH = "/kaggle/working/image_descriptions.json"
MODEL_ID = "Qwen/Qwen3.5-4B"
LIMIT = 3  # Đổi thành None để xử lý toàn bộ ảnh chưa hoàn thành
RETRY_ERRORS = False  # True để chạy lại record có status=error
MAX_NEW_TOKENS = 768
MAX_ARTICLE_CHARS = 12000  # None để dùng toàn bộ ngữ cảnh structured
PROMPT_TOKEN_WARNING = 8000
CONTEXT_FORMAT_VERSION = "structured_v1"
PROMPT_VERSION = "2.2"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
JPG_IMAGE_RE = re.compile(r"^image(\d+)\.jpg$", re.IGNORECASE)
CATEGORIES = {"SIGN_ILLUSTRATION", "TECHNICAL_DIAGRAM", "MARKING_AND_SIGNAL"}
DESCRIPTION_FIELDS = {"image_category", "summary", "items"}
ITEM_FIELDS = {"identifier", "visual_or_geometric_features", "meaning_or_specification"}


@dataclass(frozen=True)
class ImageJob:
    record_id: str
    block_id: str
    article_key: str
    document_id: str
    document_title: str
    article_id: str
    article_title: str
    article_context: str
    target_placeholder: str
    source_image: str
    inference_image: str
    image_path: Path


def load_structured_articles(path: Path) -> list[dict[str, object]]:
    articles = []
    with path.open(encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                article = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSONL at line {line_number}: {error}") from error
            if not isinstance(article, dict):
                raise ValueError(f"JSONL line {line_number} must contain an object")
            articles.append(article)
    if not articles:
        raise ValueError("Structured corpus JSONL is empty")
    return articles


def figure_placeholder(block: dict[str, object]) -> str:
    block_id = str(block.get("block_id", ""))
    filename = str(block.get("filename") or block.get("source_reference") or "")
    return f"<<FIGURE block_id={block_id} filename={filename}>>"


def iter_blocks(blocks):
    for block in blocks:
        if not isinstance(block, dict):
            raise ValueError("Every block must be an object")
        yield block
        if block.get("type") == "table":
            for cell in block.get("cells", []):
                yield from iter_blocks(cell.get("blocks", []))


def render_article_context(article: dict[str, object]) -> str:
    lines = []
    for block in iter_blocks(article.get("blocks", [])):
        block_type = block.get("type")
        if block_type == "paragraph":
            text = str(block.get("text", "")).strip()
            if text:
                lines.append(text)
        elif block_type == "figure":
            lines.append(figure_placeholder(block))
        elif block_type == "table":
            label = str(block.get("label", "")).strip()
            caption = str(block.get("caption_original", "")).strip()
            unit = str(block.get("unit_original", "")).strip()
            heading = " — ".join(value for value in (label, caption, unit) if value)
            lines.append(f"[TABLE {heading}]" if heading else "[TABLE]")
    return "\n".join(lines)


def discover_jobs(corpus_path: Path, images_dir: Path) -> list[ImageJob]:
    jobs = []
    seen_block_ids = set()
    images_root = images_dir.resolve()
    for article in load_structured_articles(corpus_path):
        required = ("article_key", "document_id", "document_title", "id", "title", "blocks")
        missing = [field for field in required if field not in article]
        if missing:
            raise ValueError(f"Article is missing required fields: {missing}")
        article_context = render_article_context(article)
        for block in iter_blocks(article["blocks"]):
            if block.get("type") != "figure":
                continue
            block_id = str(block.get("block_id", "")).strip()
            if not block_id:
                raise ValueError(f"Figure in {article['article_key']} is missing block_id")
            if block_id in seen_block_ids:
                raise ValueError(f"duplicate figure block_id: {block_id}")
            seen_block_ids.add(block_id)
            source_image = str(block.get("filename") or block.get("source_reference") or "").strip()
            if not source_image:
                raise ValueError(f"Figure {block_id} is missing filename/source_reference")
            source_path = (images_dir / source_image).resolve()
            if not source_path.is_relative_to(images_root) or not source_path.is_file():
                raise FileNotFoundError(f"Figure {block_id} image not found: {source_image}")

            inference_path = source_path
            filename_match = JPG_IMAGE_RE.fullmatch(Path(source_image).name)
            if filename_match:
                previous_number = int(filename_match.group(1)) - 1
                png_candidate = images_dir / f"image{previous_number:03d}.png"
                if png_candidate.is_file():
                    inference_path = png_candidate

            placeholder = figure_placeholder(block)
            jobs.append(ImageJob(
                record_id=block_id,
                block_id=block_id,
                article_key=str(article["article_key"]),
                document_id=str(article["document_id"]),
                document_title=str(article["document_title"]),
                article_id=str(article["id"]),
                article_title=str(article["title"]),
                article_context=article_context,
                target_placeholder=placeholder,
                source_image=source_image,
                inference_image=inference_path.name,
                image_path=inference_path,
            ))
    return jobs


def select_article_context(job: ImageJob, max_chars: int | None) -> tuple[str, bool]:
    text = job.article_context
    if max_chars is None or len(text) <= max_chars:
        return text, False
    if max_chars < len(job.target_placeholder):
        raise ValueError("MAX_ARTICLE_CHARS is too small for the target figure placeholder")
    target_start = text.find(job.target_placeholder)
    if target_start < 0:
        raise ValueError(f"Target figure placeholder is missing for {job.block_id}")
    target_end = target_start + len(job.target_placeholder)
    target_center = (target_start + target_end) // 2
    start = max(0, target_center - max_chars // 2)
    end = min(len(text), start + max_chars)
    start = max(0, end - max_chars)
    return text[start:end], True


def build_prompt(job: ImageJob) -> str:
    article_context, truncated = select_article_context(job, MAX_ARTICLE_CHARS)
    context_note = (
        "[NGỮ CẢNH ĐÃ RÚT GỌN QUANH BLOCK ẢNH MỤC TIÊU]\n"
        if truncated
        else ""
    )
    return f"""/no_think
Bạn là chuyên gia số hóa Quy chuẩn kỹ thuật quốc gia về báo hiệu đường bộ Việt Nam.

NHIỆM VỤ
Kết hợp hình ảnh được cung cấp và nội dung điều luật trong thẻ <LAW_ARTICLE> để tạo mô tả chuẩn hóa phục vụ lập chỉ mục và tìm kiếm văn bản pháp luật.

THÔNG TIN NGUỒN
- Văn bản: {job.document_id} - {job.document_title}
- Điều {job.article_id}: {job.article_title}
- Ảnh mục tiêu: {job.source_image} (block_id: {job.block_id})

<LAW_ARTICLE>
{context_note}{article_context}
</LAW_ARTICLE>

NGUYÊN TẮC CHUNG
- Chỉ dùng thông tin quan sát được trong ảnh hoặc được nêu trong điều luật.
- Điều luật là nguồn chính để xác định mã hiệu, tên gọi, ý nghĩa pháp lý và thông số kỹ thuật.
- Hình ảnh là nguồn chính để xác định hình dạng, bố cục, biểu tượng và ký hiệu nhìn thấy được.
- Không tự suy đoán mã biển, màu sắc, ý nghĩa, kích thước hoặc đơn vị đo.
- Nếu không xác định được một giá trị chuỗi, dùng chuỗi rỗng.
- Chỉ trả về một object JSON hợp lệ; không dùng Markdown và không thêm giải thích ngoài JSON.

PHÂN LOẠI
Chọn đúng một image_category:
- SIGN_ILLUSTRATION: hình mẫu một hoặc nhiều biển báo giao thông cụ thể.
- TECHNICAL_DIAGRAM: bản vẽ kỹ thuật về hình học, kích thước, góc, khoảng cách hoặc lắp đặt.
- MARKING_AND_SIGNAL: vạch kẻ đường, đèn tín hiệu, cọc tiêu, rào chắn hoặc hiệu lệnh người điều khiển giao thông.

QUY TẮC THEO LOẠI
1. SIGN_ILLUSTRATION:
- Tách từng biển thành một item.
- identifier ghi mã và tên biển theo điều luật.
- visual_or_geometric_features ghi nhóm biển, hình dạng, màu nền, màu viền và biểu tượng.
- meaning_or_specification ghi ý nghĩa báo hiệu theo điều luật.

2. TECHNICAL_DIAGRAM:
- Không mô tả màu đen/trắng của nét kỹ thuật như màu của biển thực tế.
- Không coi chữ cái kích thước là chữ in trên biển thực tế.
- Tách các hình khối hoặc bộ phận chính thành từng item.
- Liệt kê đúng các ký hiệu kích thước, góc, bán kính hoặc khoảng cách xuất hiện.
- Chỉ giải thích giá trị và ý nghĩa ký hiệu khi điều luật cung cấp căn cứ.

3. MARKING_AND_SIGNAL:
- Tách từng loại vạch, tín hiệu, thiết bị hoặc tư thế thành từng item.
- Mô tả hình thái, bố cục, hướng, vị trí, trạng thái sáng hoặc tư thế.
- Nêu tác dụng báo hiệu hoặc yêu cầu chấp hành theo điều luật.

ĐỊNH DẠNG ĐẦU RA
{{
  "image_category": "SIGN_ILLUSTRATION",
  "summary": "Một câu tóm tắt nội dung chính của hình.",
  "items": [
    {{
      "identifier": "Mã và tên biển hoặc tên đối tượng kỹ thuật",
      "visual_or_geometric_features": "Đặc điểm trực quan hoặc hình học",
      "meaning_or_specification": "Ý nghĩa báo hiệu hoặc quy định kỹ thuật"
    }}
  ]
}}

Trường image_category phải là đúng một trong ba giá trị cho phép. Không tạo thêm field.
"""


def parse_description(raw: str) -> dict[str, object]:
    object_start = raw.find("{")
    if object_start < 0:
        raise ValueError("Phản hồi không chứa JSON object")
    try:
        value, _ = json.JSONDecoder().raw_decode(raw[object_start:])
    except json.JSONDecodeError as error:
        raise ValueError(f"JSON không hợp lệ: {error}") from error

    if not isinstance(value, dict) or set(value) != DESCRIPTION_FIELDS:
        raise ValueError(f"Các field phải đúng bằng {sorted(DESCRIPTION_FIELDS)}")
    if value["image_category"] not in CATEGORIES:
        raise ValueError("image_category không hợp lệ")
    if not isinstance(value["summary"], str) or not value["summary"].strip():
        raise ValueError("summary phải là chuỗi không rỗng")
    if not isinstance(value["items"], list):
        raise ValueError("items phải là array")
    for item in value["items"]:
        if not isinstance(item, dict) or set(item) != ITEM_FIELDS:
            raise ValueError(f"Mỗi item phải có đúng các field {sorted(ITEM_FIELDS)}")
        if not all(isinstance(item[field], str) for field in ITEM_FIELDS):
            raise ValueError("Các field trong item phải là chuỗi")
    return value


def new_output(model_id: str, source_file: str) -> dict[str, object]:
    return {
        "schema_version": "2.0",
        "prompt_version": PROMPT_VERSION,
        "model_id": model_id,
        "source_file": str(source_file),
        "max_article_chars": MAX_ARTICLE_CHARS,
        "context_format_version": CONTEXT_FORMAT_VERSION,
        "records": [],
    }


def load_output(path: Path, model_id: str, source_file: str) -> dict[str, object]:
    expected = new_output(model_id, source_file)
    if not path.exists():
        return expected
    payload = json.loads(path.read_text(encoding="utf-8"))
    for field in (
        "schema_version", "prompt_version", "model_id",
        "source_file", "max_article_chars", "context_format_version",
    ):
        if payload.get(field) != expected[field]:
            raise ValueError(f"Output hiện tại không tương thích tại field {field}")
    if not isinstance(payload.get("records"), list):
        raise ValueError("records trong output phải là array")
    return payload


def prepare_output(
    output_path: Path,
    resume_path: Path | None,
    model_id: str,
    source_file: str,
) -> dict[str, object]:
    if output_path.exists():
        return load_output(output_path, model_id, source_file)
    if resume_path is not None:
        if not resume_path.is_file():
            raise FileNotFoundError(f"Không tìm thấy checkpoint: {resume_path}")
        payload = load_output(resume_path, model_id, source_file)
        save_output(output_path, payload)
        return payload
    return new_output(model_id, source_file)


def save_output(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


def append_record(path: Path, payload: dict[str, object], record: dict[str, object]) -> bool:
    record_id = record["record_id"]
    if any(existing.get("record_id") == record_id for existing in payload["records"]):
        return False
    payload["records"].append(record)
    save_output(path, payload)
    return True


def upsert_record(path: Path, payload: dict[str, object], record: dict[str, object]) -> None:
    for index, existing in enumerate(payload["records"]):
        if existing.get("record_id") == record["record_id"]:
            payload["records"][index] = record
            save_output(path, payload)
            return
    payload["records"].append(record)
    save_output(path, payload)


def make_result_record(
    job: ImageJob,
    status: str,
    description: dict[str, object] | None,
    error: str,
) -> dict[str, object]:
    return {
        "record_id": job.record_id,
        "block_id": job.block_id,
        "article_key": job.article_key,
        "document_id": job.document_id,
        "document_title": job.document_title,
        "article_id": job.article_id,
        "article_title": job.article_title,
        "source_image": job.source_image,
        "inference_image": job.inference_image,
        "status": status,
        "error": error,
        "description": description,
    }


def run_jobs(
    jobs: list[ImageJob],
    output_path: Path,
    payload: dict[str, object],
    describe,
    limit: int | None,
    retry_errors: bool,
    on_error=None,
) -> dict[str, int]:
    completed = set()
    for record in payload["records"]:
        if retry_errors and record.get("status") == "error":
            continue
        completed.add(record["record_id"])

    pending = [job for job in jobs if job.record_id not in completed]
    if limit is not None:
        pending = pending[:limit]

    save_output(output_path, payload)
    stats = {"attempted": 0, "succeeded": 0, "failed": 0}
    total = len(pending)
    for current, job in enumerate(pending, start=1):
        print(f"[{current}/{total}] {job.record_id}", flush=True)
        stats["attempted"] += 1
        try:
            description = describe(job)
            record = make_result_record(job, "success", description, "")
            stats["succeeded"] += 1
        except Exception as exception:
            error_message = f"{type(exception).__name__}: {exception}"
            record = make_result_record(job, "error", None, error_message)
            stats["failed"] += 1
            print(f"  Error: {error_message}", flush=True)
            if on_error is not None:
                on_error(job, exception)
        upsert_record(output_path, payload, record)
    return stats

## 2. Kiểm tra input và lập danh sách ảnh

Cell này dừng với thông báo rõ ràng nếu đường dẫn JSONL hoặc thư mục ảnh chưa được cấu hình. Mọi block `figure`, kể cả ảnh nằm trong ô bảng, được lập thành một job có khóa `block_id`.

In [ ]:
if not STRUCTURED_CORPUS_PATH.strip() or not IMAGES_DIR.strip():
    raise ValueError("Hãy điền STRUCTURED_CORPUS_PATH và IMAGES_DIR trong ô cấu hình trước khi chạy.")

structured_corpus_path = Path(STRUCTURED_CORPUS_PATH)
images_dir = Path(IMAGES_DIR)
resume_path = Path(RESUME_PATH) if RESUME_PATH.strip() else None
output_path = Path(OUTPUT_PATH)

if not structured_corpus_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy structured corpus: {structured_corpus_path}")
if not images_dir.is_dir():
    raise NotADirectoryError(f"Không tìm thấy thư mục ảnh: {images_dir}")

jobs = discover_jobs(structured_corpus_path, images_dir)
print(f"Đã tìm thấy {len(jobs)} block figure trong structured corpus.")
print(f"Output độc lập: {output_path}")

## 3. Nạp VLM

Lần chạy đầu có thể mất thời gian để tải model. T4/P100 dùng `float16`; GPU hỗ trợ tốt BF16 sẽ tự dùng `bfloat16`.

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("Notebook cần Kaggle GPU. Hãy bật Accelerator trong Settings.")

major_capability, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major_capability >= 8 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.eval()
print(f"Đã nạp {MODEL_ID} trên {torch.cuda.get_device_name(0)} với {compute_dtype}.")

## 4. Hàm inference và chạy pipeline

Mỗi phản hồi sai schema được thử lại đúng một lần. Sau mỗi ảnh, toàn bộ JSON được ghi atomically vào `OUTPUT_PATH`. Ảnh lỗi được ghi với `status=error` và pipeline tiếp tục xử lý ảnh kế tiếp.

In [ ]:
def describe_with_qwen(job: ImageJob) -> dict[str, object]:
    base_prompt = build_prompt(job)
    prompt_token_count = len(
        processor.tokenizer(base_prompt, add_special_tokens=False)["input_ids"]
    )
    if prompt_token_count > PROMPT_TOKEN_WARNING:
        print(
            f"  Cảnh báo: prompt có khoảng {prompt_token_count:,} text token; "
            "ảnh này có thể xử lý chậm hoặc thiếu VRAM.",
            flush=True,
        )
    last_error = None

    for attempt in range(2):
        prompt = base_prompt
        if attempt == 1:
            prompt += (
                "\nPhản hồi trước không phải JSON đúng schema. "
                "Hãy trả lại đúng một object JSON hợp lệ và không thêm nội dung khác."
            )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "path": str(job.image_path.resolve())},
                {"type": "text", "text": prompt},
            ],
        }]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
            )
        input_length = inputs["input_ids"].shape[1]
        generated_only = generated_ids[:, input_length:]
        raw_response = processor.batch_decode(
            generated_only,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        try:
            return parse_description(raw_response)
        except ValueError as error:
            last_error = error

    raise ValueError(f"Model trả JSON sai schema sau 2 lần: {last_error}")

In [ ]:
def release_cuda_after_error(job: ImageJob, exception: Exception) -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


payload = prepare_output(
    output_path, resume_path, MODEL_ID, str(structured_corpus_path)
)
print(f"Output hiện có {len(payload['records'])} record.")
stats = run_jobs(
    jobs,
    output_path,
    payload,
    describe_with_qwen,
    limit=LIMIT,
    retry_errors=RETRY_ERRORS,
    on_error=release_cuda_after_error,
)

print(f"Kết quả lần chạy: {stats}")
print(f"Tổng số record trong file: {len(payload['records'])}")
print(f"File kết quả: {output_path}")

## 5. Kiểm tra và tải output

Kaggle hiển thị file JSON độc lập trong thư mục **Output** sau khi lưu phiên notebook. Cell dưới đây kiểm tra nhanh metadata và record cuối cùng; structured corpus đầu vào không bị sửa đổi.

In [ ]:
result = json.loads(output_path.read_text(encoding="utf-8"))
print(json.dumps({
    "schema_version": result["schema_version"],
    "model_id": result["model_id"],
    "record_count": len(result["records"]),
}, ensure_ascii=False, indent=2))

if result["records"]:
    print(json.dumps(result["records"][-1], ensure_ascii=False, indent=2))